# Non-Stationary Bandits: When the Best Option Changes

**Goal:** Understand why standard bandits fail in changing environments and how adaptive algorithms handle regime shifts.

**Key Question:** What happens when arm 1 is best for 500 steps, then arm 2 becomes best, then arm 3?

**Real-world example:** In commodity markets, regime changes are the rule. COVID crash made oil terrible. 2022 energy crisis made natural gas great. Standard bandits get stuck on old winners.

## Setup: Create a Bandit with Regime Shifts

We'll simulate a 3-armed bandit where the best arm changes at steps 500 and 1500.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta
import pandas as pd

np.random.seed(42)

def get_reward_probabilities(t):
    """Reward probabilities change at t=500 and t=1500."""
    if t < 500:
        return [0.7, 0.4, 0.3]  # Arm 0 is best
    elif t < 1500:
        return [0.3, 0.7, 0.4]  # Arm 1 becomes best
    else:
        return [0.2, 0.3, 0.8]  # Arm 2 becomes best

# Visualize the regime structure
T = 2000
probs = np.array([get_reward_probabilities(t) for t in range(T)])

plt.figure(figsize=(12, 4))
for i in range(3):
    plt.plot(probs[:, i], label=f'Arm {i}', linewidth=2)
plt.axvline(500, color='red', linestyle='--', alpha=0.5, label='Regime change')
plt.axvline(1500, color='red', linestyle='--', alpha=0.5)
plt.xlabel('Time Step')
plt.ylabel('Reward Probability')
plt.title('Non-Stationary Bandit: Reward Probabilities Over Time')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Regime 1 (0-499): Arm 0 is best (p=0.7)")
print("Regime 2 (500-1499): Arm 1 is best (p=0.7)")
print("Regime 3 (1500-1999): Arm 2 is best (p=0.8)")

## Standard Thompson Sampling: Fails to Adapt

Let's run standard Thompson Sampling and watch it get stuck on old winners.

In [ ]:
class StandardThompsonSampling:
    def __init__(self, n_arms):
        self.n_arms = n_arms
        self.alpha = np.ones(n_arms)
        self.beta_params = np.ones(n_arms)

    def select_arm(self):
        samples = [beta.rvs(a, b) for a, b in zip(self.alpha, self.beta_params)]
        return np.argmax(samples)

    def update(self, arm, reward):
        self.alpha[arm] += reward
        self.beta_params[arm] += (1 - reward)

# Run standard Thompson Sampling
bandit_std = StandardThompsonSampling(n_arms=3)
selections_std = []
rewards_std = []

for t in range(T):
    arm = bandit_std.select_arm()
    selections_std.append(arm)

    # Get reward from current regime
    probs = get_reward_probabilities(t)
    reward = np.random.binomial(1, probs[arm])
    rewards_std.append(reward)

    bandit_std.update(arm, reward)

# Plot selections over time
plt.figure(figsize=(12, 4))
plt.scatter(range(T), selections_std, alpha=0.3, s=1)
plt.axvline(500, color='red', linestyle='--', alpha=0.5, label='Regime change')
plt.axvline(1500, color='red', linestyle='--', alpha=0.5)
plt.xlabel('Time Step')
plt.ylabel('Selected Arm')
plt.title('Standard Thompson Sampling: Gets Stuck on Old Winners')
plt.yticks([0, 1, 2])
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Cumulative reward: {sum(rewards_std)}")
print("Notice: Algorithm keeps selecting Arm 0 long after it becomes suboptimal!")

## Discounted Thompson Sampling: Adapts via Exponential Forgetting

Discount old observations exponentially — recent data matters more.

In [ ]:
class DiscountedThompsonSampling:
    def __init__(self, n_arms, gamma=0.95):
        self.n_arms = n_arms
        self.gamma = gamma
        self.alpha = np.ones(n_arms)
        self.beta_params = np.ones(n_arms)

    def select_arm(self):
        samples = [beta.rvs(a, b) for a, b in zip(self.alpha, self.beta_params)]
        return np.argmax(samples)

    def update(self, arm, reward):
        # Discount all arms
        self.alpha *= self.gamma
        self.beta_params *= self.gamma

        # Update selected arm
        self.alpha[arm] += reward
        self.beta_params[arm] += (1 - reward)

# Run discounted Thompson Sampling
bandit_disc = DiscountedThompsonSampling(n_arms=3, gamma=0.95)
selections_disc = []
rewards_disc = []

for t in range(T):
    arm = bandit_disc.select_arm()
    selections_disc.append(arm)

    probs = get_reward_probabilities(t)
    reward = np.random.binomial(1, probs[arm])
    rewards_disc.append(reward)

    bandit_disc.update(arm, reward)

# Plot selections
plt.figure(figsize=(12, 4))
plt.scatter(range(T), selections_disc, alpha=0.3, s=1)
plt.axvline(500, color='red', linestyle='--', alpha=0.5, label='Regime change')
plt.axvline(1500, color='red', linestyle='--', alpha=0.5)
plt.xlabel('Time Step')
plt.ylabel('Selected Arm')
plt.title('Discounted Thompson Sampling (γ=0.95): Adapts Quickly')
plt.yticks([0, 1, 2])
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Cumulative reward: {sum(rewards_disc)}")
print("Notice: Algorithm switches to new best arm within ~50-100 steps after each regime change!")

## Sliding-Window UCB: Hard Cutoff on Old Data

Only use the last W observations — completely forget older data.

In [ ]:
from collections import deque

class SlidingWindowUCB:
    def __init__(self, n_arms, window_size=200):
        self.n_arms = n_arms
        self.window_size = window_size
        self.history = deque(maxlen=window_size)
        self.t = 0

    def select_arm(self):
        self.t += 1

        counts = np.zeros(self.n_arms)
        sums = np.zeros(self.n_arms)
        for arm, reward in self.history:
            counts[arm] += 1
            sums[arm] += reward

        ucb_values = []
        for i in range(self.n_arms):
            if counts[i] == 0:
                ucb_values.append(float('inf'))
            else:
                mean = sums[i] / counts[i]
                exploration = np.sqrt(2 * np.log(self.t) / counts[i])
                ucb_values.append(mean + exploration)

        return np.argmax(ucb_values)

    def update(self, arm, reward):
        self.history.append((arm, reward))

# Run sliding-window UCB
bandit_sw = SlidingWindowUCB(n_arms=3, window_size=200)
selections_sw = []
rewards_sw = []

for t in range(T):
    arm = bandit_sw.select_arm()
    selections_sw.append(arm)

    probs = get_reward_probabilities(t)
    reward = np.random.binomial(1, probs[arm])
    rewards_sw.append(reward)

    bandit_sw.update(arm, reward)

# Plot selections
plt.figure(figsize=(12, 4))
plt.scatter(range(T), selections_sw, alpha=0.3, s=1)
plt.axvline(500, color='red', linestyle='--', alpha=0.5, label='Regime change')
plt.axvline(1500, color='red', linestyle='--', alpha=0.5)
plt.xlabel('Time Step')
plt.ylabel('Selected Arm')
plt.title('Sliding-Window UCB (W=200): Adapts via Hard Cutoff')
plt.yticks([0, 1, 2])
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Cumulative reward: {sum(rewards_sw)}")
print("Notice: Algorithm adapts differently than discounted TS — more aggressive exploration")

## Compare All Three: Cumulative Regret

Regret = total reward you could have gotten (with oracle knowledge) minus what you actually got.

In [ ]:
# Compute optimal rewards (oracle that knows regime structure)
optimal_rewards = []
for t in range(T):
    probs = get_reward_probabilities(t)
    best_arm = np.argmax(probs)
    optimal_rewards.append(np.random.binomial(1, probs[best_arm]))

# Compute cumulative regret for each algorithm
regret_std = np.cumsum(optimal_rewards) - np.cumsum(rewards_std)
regret_disc = np.cumsum(optimal_rewards) - np.cumsum(rewards_disc)
regret_sw = np.cumsum(optimal_rewards) - np.cumsum(rewards_sw)

plt.figure(figsize=(12, 5))
plt.plot(regret_std, label='Standard TS (fails)', linewidth=2, alpha=0.8)
plt.plot(regret_disc, label='Discounted TS (γ=0.95)', linewidth=2, alpha=0.8)
plt.plot(regret_sw, label='Sliding-Window UCB (W=200)', linewidth=2, alpha=0.8)
plt.axvline(500, color='red', linestyle='--', alpha=0.3)
plt.axvline(1500, color='red', linestyle='--', alpha=0.3)
plt.xlabel('Time Step')
plt.ylabel('Cumulative Regret')
plt.title('Regret Comparison: Adaptive Algorithms Minimize Regret After Regime Changes')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Final regret - Standard TS: {regret_std[-1]:.0f}")
print(f"Final regret - Discounted TS: {regret_disc[-1]:.0f}")
print(f"Final regret - Sliding-Window UCB: {regret_sw[-1]:.0f}")
print("\nKey insight: Adaptive algorithms have much lower regret!")

## Modify This: Experiment with Parameters

Try different scenarios and hyperparameters.

In [ ]:
# MODIFY THESE PARAMETERS
gamma_values = [0.90, 0.95, 0.99]  # Try different discount factors
window_sizes = [100, 200, 500]     # Try different window sizes

# Compare different gammas
plt.figure(figsize=(12, 5))

for gamma in gamma_values:
    bandit = DiscountedThompsonSampling(n_arms=3, gamma=gamma)
    rewards = []

    for t in range(T):
        arm = bandit.select_arm()
        probs = get_reward_probabilities(t)
        reward = np.random.binomial(1, probs[arm])
        rewards.append(reward)
        bandit.update(arm, reward)

    regret = np.cumsum(optimal_rewards) - np.cumsum(rewards)
    plt.plot(regret, label=f'γ={gamma}', linewidth=2, alpha=0.7)

plt.axvline(500, color='red', linestyle='--', alpha=0.3, label='Regime changes')
plt.axvline(1500, color='red', linestyle='--', alpha=0.3)
plt.xlabel('Time Step')
plt.ylabel('Cumulative Regret')
plt.title('Discounted TS: Effect of γ on Adaptation Speed')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Observations:")
print("- Lower γ (0.90) → faster adaptation, higher variance")
print("- Higher γ (0.99) → slower adaptation, more stable")
print("- Trade-off: speed vs stability")

## Summary

**What we learned:**

1. **Standard bandits fail in non-stationary environments** — they accumulate too much historical evidence and get stuck

2. **Discounted Thompson Sampling** adapts via exponential forgetting (γ parameter controls speed)

3. **Sliding-Window UCB** adapts via hard cutoff (window size W determines memory)

4. **Hyperparameter tuning matters:**
   - Lower γ or smaller W → faster adaptation but higher variance
   - Higher γ or larger W → slower adaptation but more stable

5. **Commodity trading insight:** Markets shift regimes constantly (COVID, energy crisis, inflation). Non-stationary algorithms are essential.

**Next steps:**
- Notebook 2: Change-point detection to actively detect regime shifts
- Notebook 3: Apply these algorithms to real commodity data
- Module 6 Guide 2: Restless bandits (when unselected arms also evolve)